In [1]:
import csv
import os
from datetime import datetime




class MenuItem:
    def __init__(self, item_id, name, category, price):
        self.item_id = item_id
        self.name = name
        self.category = category
        self.price = price

    def display(self):
        print(f"{self.item_id:<8}{self.name:<20}{self.category:<15}₹{self.price:.2f}")




class Restaurant:
    def __init__(self):
        self.menu = []
        self.orders = {}
        self.order_id = 1001


        self.categories = ("Starters", "Main Course", "Beverages", "Desserts")


        self.available_categories = set()


        self.load_menu()
        self.load_orders()



    def load_menu(self):
        self.menu = [
            MenuItem(1, "French Fries", "Starters", 120),
            MenuItem(2, "Chicken Burger", "Main Course", 180),
            MenuItem(3, "Veg Pizza", "Main Course", 220),
            MenuItem(4, "Pasta", "Main Course", 160),
            MenuItem(5, "Fresh Lime", "Beverages", 80),
            MenuItem(6, "Cold Coffee", "Beverages", 100),
            MenuItem(7, "Ice Cream", "Desserts", 90),
            MenuItem(8, "Chocolate Cake", "Desserts", 130)
        ]

        for item in self.menu:
            self.available_categories.add(item.category)



    def display_menu(self):
        print("\n================ RESTAURANT MENU ================")
        print(f"{'ID':<8}{'Item':<20}{'Category':<15}{'Price'}")
        print("--------------------------------------------------")

        for item in self.menu:
            item.display()

        print("==================================================")


    def search_food(self):
        keyword = input("Enter food name to search: ").strip().lower()

        found = False

        for item in self.menu:
            if keyword in item.name.lower():
                item.display()
                found = True

        if not found:
            print("No matching food item found.")



    def place_order(self):
        self.display_menu()

        cart = []
        total = 0

        while True:
            try:
                item_id = int(input("\nEnter Item ID (0 to finish): "))

                if item_id == 0:
                    break

                selected_item = None

                for item in self.menu:
                    if item.item_id == item_id:
                        selected_item = item
                        break

                if selected_item is None:
                    print("Invalid Item ID.")
                    continue

                quantity = int(input("Enter quantity: "))

                if quantity <= 0:
                    print("Quantity must be greater than zero.")
                    continue

                amount = selected_item.price * quantity


                order_item = {
                    "name": selected_item.name,
                    "quantity": quantity,
                    "price": selected_item.price,
                    "amount": amount
                }

                cart.append(order_item)
                total += amount

                print(
                    f"{quantity} x {selected_item.name} "
                    f"added to cart."
                )

            except ValueError:
                print("Invalid input! Please enter numbers only.")

        if not cart:
            print("No items selected. Order cancelled.")
            return


        discount = 0

        if total >= 1000:
            discount = total * 0.10
        elif total >= 500:
            discount = total * 0.05

        final_amount = total - discount

        order_number = self.order_id
        self.order_id += 1

        order = {
            "items": cart,
            "total": total,
            "discount": discount,
            "final_amount": final_amount,
            "date": datetime.now().strftime("%d-%m-%Y %H:%M:%S")
        }

        self.orders[order_number] = order

        self.save_order(order_number, order)
        self.display_bill(order_number, order)



    def display_bill(self, order_number, order):
        print("\n================= BILL =================")

        print(f"Order ID : {order_number}")
        print(f"Date     : {order['date']}")

        print("-----------------------------------------")
        print(f"{'Item':<20}{'Qty':<8}{'Amount'}")
        print("-----------------------------------------")

        for item in order["items"]:
            print(
                f"{item['name']:<20}"
                f"{item['quantity']:<8}"
                f"₹{item['amount']:.2f}"
            )

        print("-----------------------------------------")
        print(f"Subtotal       : ₹{order['total']:.2f}")
        print(f"Discount       : ₹{order['discount']:.2f}")
        print(f"Final Amount   : ₹{order['final_amount']:.2f}")
        print("=========================================")



    def save_order(self, order_number, order):
        file_exists = os.path.exists("orders.csv")

        with open("orders.csv", "a", newline="") as file:
            writer = csv.writer(file)

            if not file_exists:
                writer.writerow([
                    "Order ID",
                    "Date",
                    "Items",
                    "Subtotal",
                    "Discount",
                    "Final Amount"
                ])

            items = ", ".join(
                f"{item['name']} x {item['quantity']}"
                for item in order["items"]
            )

            writer.writerow([
                order_number,
                order["date"],
                items,
                order["total"],
                order["discount"],
                order["final_amount"]
            ])



    def load_orders(self):
        if not os.path.exists("orders.csv"):
            return

        try:
            with open("orders.csv", "r") as file:
                reader = csv.DictReader(file)

                for row in reader:
                    try:
                        order_id = int(row["Order ID"])

                        self.orders[order_id] = {
                            "items": row["Items"],
                            "total": float(row["Subtotal"]),
                            "discount": float(row["Discount"]),
                            "final_amount": float(row["Final Amount"]),
                            "date": row["Date"]
                        }

                        if order_id >= self.order_id:
                            self.order_id = order_id + 1

                    except (ValueError, KeyError):
                        continue

        except FileNotFoundError:
            pass


    def view_orders(self):
        if not self.orders:
            print("No orders found.")
            return

        print("\n================ ORDER HISTORY ================")

        for order_id, order in self.orders.items():
            print(f"\nOrder ID : {order_id}")
            print(f"Date     : {order['date']}")
            print(f"Items    : {order['items']}")
            print(f"Subtotal : ₹{order['total']:.2f}")
            print(f"Discount : ₹{order['discount']:.2f}")
            print(f"Total    : ₹{order['final_amount']:.2f}")

        print("================================================")



    def view_categories(self):
        print("\nAvailable Categories:")

        for category in self.available_categories:
            print("-", category)



    def cancel_order(self):
        try:
            order_id = int(input("Enter Order ID to cancel: "))

            if order_id in self.orders:
                del self.orders[order_id]

                print("Order cancelled successfully.")
                print("Note: The cancellation is applied to the current session.")

            else:
                print("Order ID not found.")

        except ValueError:
            print("Invalid Order ID. Please enter a number.")


    def total_sales(self):
        if not self.orders:
            print("No sales available.")
            return

        total = 0

        for order in self.orders.values():
            if isinstance(order["final_amount"], (int, float)):
                total += order["final_amount"]

        print(f"\nTotal Sales: ₹{total:.2f}")




def main():
    restaurant = Restaurant()

    while True:

        print("\n")
        print("==============================================")
        print("       RESTAURANT ORDERING SYSTEM")
        print("==============================================")
        print("1. Display Menu")
        print("2. Search Food")
        print("3. View Categories")
        print("4. Place Order")
        print("5. View Order History")
        print("6. Cancel Order")
        print("7. View Total Sales")
        print("8. Exit")
        print("==============================================")

        choice = input("Enter your choice: ").strip()

        if choice == "1":
            restaurant.display_menu()

        elif choice == "2":
            restaurant.search_food()

        elif choice == "3":
            restaurant.view_categories()

        elif choice == "4":
            restaurant.place_order()

        elif choice == "5":
            restaurant.view_orders()

        elif choice == "6":
            restaurant.cancel_order()

        elif choice == "7":
            restaurant.total_sales()

        elif choice == "8":
            print("\nThank you for using the Restaurant Ordering System!")
            print("Visit again!")
            break

        else:
            print("Invalid choice! Please select a number from 1 to 8.")



main()



       RESTAURANT ORDERING SYSTEM
1. Display Menu
2. Search Food
3. View Categories
4. Place Order
5. View Order History
6. Cancel Order
7. View Total Sales
8. Exit
Enter your choice: 2
Enter food name to search: pasta
4       Pasta               Main Course    ₹160.00


       RESTAURANT ORDERING SYSTEM
1. Display Menu
2. Search Food
3. View Categories
4. Place Order
5. View Order History
6. Cancel Order
7. View Total Sales
8. Exit
Enter your choice: 1

================ RESTAURANT MENU ================
ID      Item                Category       Price
--------------------------------------------------
1       French Fries        Starters       ₹120.00
2       Chicken Burger      Main Course    ₹180.00
3       Veg Pizza           Main Course    ₹220.00
4       Pasta               Main Course    ₹160.00
5       Fresh Lime          Beverages      ₹80.00
6       Cold Coffee         Beverages      ₹100.00
7       Ice Cream           Desserts       ₹90.00
8       Chocolate Cake      Desser